# MicroFarmCast Model 1 — setup and data discovery
This notebook starts with environment and file discovery. It does not download data or assume local datasets exist. After discovery, choose a study area and date range before requesting or downloading source data.

In [ ]:
from pathlib import Path
import importlib.util
import platform
import sys

ROOT = Path.cwd()
print("Python:", sys.version.split()[0], "|", platform.platform())
print("Notebook working directory:", ROOT.resolve())

packages = ["pandas", "numpy", "xgboost", "sklearn", "matplotlib", "jupyterlab", "xarray", "netCDF4", "rasterio", "cdsapi", "geopandas"]
print("\\nPackage availability:")
for package in packages:
    print(f"{package:12} {\"installed\" if importlib.util.find_spec(package) else \"MISSING\"}")

# Look for likely source files without scanning the entire computer.
data_roots = [ROOT / "data", ROOT / "datasets", ROOT]
extensions = {".csv", ".parquet", ".nc", ".grib", ".grb", ".tif", ".tiff", ".xlsx", ".zip"}
found = set()
for folder in data_roots:
    if folder.exists():
        found.update(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in extensions)
print("\\nCandidate data files:")
if found:
    for p in sorted(found): print(" -", p.relative_to(ROOT) if p.is_relative_to(ROOT) else p)
else:
    print("None found in data/, datasets/, or the notebook workspace.")

## Data plan (no downloads yet)

- **ERA5-Land hourly**: historical land weather and soil variables from the Copernicus Climate Data Store. The dataset is hourly, global land coverage, with a grid around 9 km; it is reanalysis, not farm-scale sensor truth. Review variable definitions/units and the current CDS licence and access terms before use. CDS account/API credentials may be required for programmatic retrieval.
- **Copernicus DEM GLO-30**: elevation terrain context at roughly 30 m, subject to current access eligibility and licence terms. Slope/aspect can be derived from a local DEM tile with a geospatial raster library; its scale does not make ERA5 weather 30 m resolution.
- **Local station/farm sensors**: needed for credible farm-level target labels and evaluation. Without them, outputs must be labelled a coarse-grid prototype and must not be described as validated 100–500 m farm forecasts.

Access references: [ERA5-Land dataset and licence](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land), [Copernicus DEM collection](https://dataspace.copernicus.eu/explore-data/data-collections/copernicus-contributing-missions/collections-description/COP-DEM), [Copernicus DEM access documentation](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Data/DEM.html).

## Next decision

Before retrieval, specify: (1) study area or farm coordinates/boundary and country, (2) historical start/end dates, (3) whether local sensor data exist and their format/time zone, and (4) whether you can use a CDS account/API key and access Copernicus DEM GLO-30. I will then tailor the data requests and keep each next stage reviewable.